# DRACO Benchmark with ScreamingFace 😱


Deep Research Accuracy, Completeness, and Objectivity (DRACO) Benchmark is an open benchmark for
evaluating deep research agents grounded in how users actually use AI for complex research tasks
published by Perplexity
([blog post](https://research.perplexity.ai/articles/evaluating-deep-research-performance-in-the-wild-with-the-draco-benchmark),
[paper](https://arxiv.org/pdf/2602.11685).
DRACO consists of 100 research tasks, each paired with expert crafted rubrics
averaging ~40 evaluation criteria.

This notebook evaluates DRACO using new models (August 2026) and fusions of these models on
[screamingface](https://github.com/ScreamingFace/screamingface).

<img src="assets/draco-benchmark.svg" width="900"
  alt="DRACO at a glance: 100 research tasks, ~40 weighted rubric criteria each, judge
  answers MET/UNMET per criterion, score = mean of weighted case scores in 0..1"/>

## Running things locally

From a terminal:

```bash
screamingface prepare draco  # first run only: download pinned Benchmark assets
screamingface up             # start Gateway :9105, Scoreboard :9106, and Engine :9108
screamingface status
```

Use `screamingface logs` to inspect startup failures and `screamingface down` when finished. Stack
management stays outside the notebook so **Run All** never starts or stops local services.

Export `TAVILY_API_KEY` before `screamingface up`: the Gemini, Kimi, DeepSeek, and
Qwen answer routes use its guarded tool loop, and the Engine fails before model spend when that
required retrieval mechanism is unavailable.

In [ ]:
import screamingface as sf

## 1. Connect OpenRouter

In [ ]:
sf.connect()

## 2. Define the models

In [ ]:
ANSWER_PROMPT = (
    "You are answering a research-quality prompt. Provide a thorough, "
    "well-reasoned answer in prose. Address every aspect the prompt raises. "
    "Use clear structure (headings, bullet lists where appropriate) and cite "
    "specific facts, methodologies, or sources where relevant.\n\n"
    "Do not refuse, abstain, or claim uncertainty unless the question is "
    "genuinely ambiguous — the goal is to demonstrate depth of understanding. "
    "Length: aim for the level of detail the question warrants; brevity that "
    "skips key points will be penalised by the rubric."
)

In [ ]:
PARAMS = {"max_tokens": 32768, "temperature": 0.0}

deepseek = sf.Model(
    model="openrouter/deepseek/deepseek-v4-pro",
    prompt=ANSWER_PROMPT,
    params=PARAMS,
)
qwen = sf.Model(
    model="openrouter/qwen/qwen3-coder",
    prompt=ANSWER_PROMPT,
    params=PARAMS,
)
glm = sf.Model(
    model="openrouter/z-ai/glm-5.2",
    prompt=ANSWER_PROMPT,
    params=PARAMS,
)

## 3. Define the synthesize model and the fusion

In [ ]:
SYNTHESIS_PROMPT = (
    "You are synthesising a single, comprehensive answer to a research-quality "
    "prompt by combining N independent answers from a panel of models. The "
    "downstream grader will score your output against a STRUCTURED RUBRIC of "
    "weighted criteria — your goal is to maximise rubric coverage.\n\n"
    "Procedure:\n"
    "1. Read every panel answer carefully.\n"
    "2. Identify which claims, facts, citations, or arguments each panel member "
    "contributes that the others miss.\n"
    "3. Produce ONE unified prose response that:\n"
    "   - Combines the strongest reasoning from every panel member\n"
    "   - Preserves specific named entities, dates, methodologies, and citations\n"
    "   - Resolves disagreements by favouring the more specific / better-cited claim\n"
    "   - Uses clear structure (headings, lists) where it aids the reader\n"
    "4. Do not introduce new facts that no panel member provided.\n"
    "5. Do not hedge or refuse — the panel collectively has enough material.\n\n"
    "Output: the unified prose answer, no preamble, no JSON wrapper."
)

kimi = sf.Model(
    model="openrouter/moonshotai/kimi-k3",
    prompt=SYNTHESIS_PROMPT,
    params=PARAMS,
)

best_open_source = sf.Fusion(
    members=[deepseek, glm, qwen], name="best_open_source", synthesizer=kimi
)

## 3. Run DRACO with the fusion

In [ ]:
report = sf.evaluate(best_open_source, benchmark="draco", limit=1)
report

## 4. Send the score to the Scoreboard

Publication takes the evaluated `CandidateResult` and submits the Benchmark's **native
score** exactly as the Engine graded it — fractional or negative values included — and the
Scoreboard stores and ranks it without recalculating. Opt-in so **Run All** never changes
the public Leaderboard.

In [ ]:
PUBLISH_RESULT = False

submission = sf.leaderboards.submit(report.candidates.only) if PUBLISH_RESULT else None
submission